<a href="https://colab.research.google.com/github/junejack0531-commits/-/blob/main/%ED%95%B4%EC%82%AC%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%A7%88%EC%9D%B4%EB%8B%9D%2013%EC%A3%BC%EC%B0%A8%20%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR = Path('/content/type3')  # 필요시 수정
df = pd.read_csv(DATA_DIR / '/content/sample_data/churn_logit1.csv')

print("=== 데이터 미리보기 ===")
print(df.head())
print("\n=== 기본 정보 ===")
print(df.info())

# 로지스틱 회귀 적합
model = smf.logit('churn ~ age + usage_hour + complaint_cnt', data=df).fit()

print("\n=== 로지스틱 회귀 요약 ===")
print(model.summary())

# 계수표 정리
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)
})

print("\n=== 계수 / p-value / 오즈비 ===")
print(result_table)

# 95% 신뢰구간과 오즈비 기준으로도 보기
conf = model.conf_int() #interval
conf.columns = ['2.5%', '97.5%']
conf['OR_2.5%'] = np.exp(conf['2.5%'])   #odd비는 exp
conf['OR_97.5%'] = np.exp(conf['97.5%'])

print("\n=== 계수 신뢰구간 및 오즈비 신뢰구간 ===")
print(conf)

# 예측확률 및 분류
df['pred_prob'] = model.predict(df)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int) #결과를 정수로

print("\n=== 예측확률 상위 5개 ===")
print(df[['churn', 'pred_prob', 'pred_class']].head())

# 해석용 출력
print("\n=== 해석 가이드 ===")
for var in ['age', 'usage_hour', 'complaint_cnt']:
    coef = model.params[var]
    pval = model.pvalues[var]
    or_val = np.exp(coef)
    direction = '증가' if coef > 0 else '감소'
    print(f"{var}: 계수={coef:.4f}, p-value={pval:.6f}, 오즈비={or_val:.4f}")
    print(f" -> {var}가 1단위 증가할 때 이탈 odds는 약 {or_val:.4f}배가 되며, 방향은 {direction}입니다.")


=== 데이터 미리보기 ===
   churn  age  usage_hour  complaint_cnt
0      0   29       12.07              0
1      0   20       70.05              2
2      0   23       58.09              0
3      0   37       67.25              2
4      0   54       78.19              0

=== 기본 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   churn          500 non-null    int64  
 1   age            500 non-null    int64  
 2   usage_hour     500 non-null    float64
 3   complaint_cnt  500 non-null    int64  
dtypes: float64(1), int64(3)
memory usage: 15.8 KB
None
Optimization terminated successfully.
         Current function value: 0.237853
         Iterations 7

=== 로지스틱 회귀 요약 ===
                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                  500
Model:            

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score
from pathlib import Path

DATA_DIR = Path('/content/type3')  # 필요시 수정
df = pd.read_csv(DATA_DIR / '/content/sample_data/loan_default_logit2.csv')

print("=== 데이터 미리보기 ===")
print(df.head())
print("\n=== 기본 정보 ===")
print(df.info())

# 로지스틱 회귀 적합
model = smf.logit('default ~ income + debt_ratio + late_cnt', data=df).fit()

print("\n=== 로지스틱 회귀 요약 ===")
print(model.summary())

# 계수 / p-value / 오즈비
result_table = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)   #오즈비는 exp
})
print("\n=== 계수 / p-value / 오즈비 ===")
print(result_table)

# 로그우도와 residual deviance
llf = model.llf
residual_deviance = -2 * llf

print("\n=== 적합도 지표 ===")
print(f"log-likelihood (llf): {llf:.4f}")
print(f"residual deviance: {residual_deviance:.4f}")

# 예측확률, 예측분류
df['pred_prob'] = model.predict(df)
df['pred_class'] = (df['pred_prob'] >= 0.5).astype(int) #정수로 변환

# 성능지표
acc = accuracy_score(df['default'], df['pred_class'])
err = 1 - acc

print("\n=== 분류 성능 ===")
print(f"accuracy     : {acc:.4f}")
print(f"error rate   : {err:.4f}")

# 예측확률 일부 확인
print("\n=== 예측확률 상위 10개 ===")
print(df[['default', 'pred_prob', 'pred_class']].head(10))

# 신뢰구간
conf = model.conf_int()
conf.columns = ['2.5%', '97.5%']
conf['OR_2.5%'] = np.exp(conf['2.5%'])
conf['OR_97.5%'] = np.exp(conf['97.5%'])

print("\n=== 계수 신뢰구간 및 오즈비 신뢰구간 ===")
print(conf)

# 해석용 출력
print("\n=== 해석 가이드 ===")
for var in ['income', 'debt_ratio', 'late_cnt']:
    coef = model.params[var]
    pval = model.pvalues[var]
    or_val = np.exp(coef)
    print(f"{var}: 계수={coef:.6f}, p-value={pval:.6f}, 오즈비={or_val:.6f}")


=== 데이터 미리보기 ===
   default   income  debt_ratio  late_cnt
0        0  3313.39       0.265         2
1        0  4026.60       0.305         0
2        0  3262.68       0.684         2
3        0  4087.12       1.031         0
4        0  2397.14       0.178         0

=== 기본 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   default     600 non-null    int64  
 1   income      600 non-null    float64
 2   debt_ratio  600 non-null    float64
 3   late_cnt    600 non-null    int64  
dtypes: float64(2), int64(2)
memory usage: 18.9 KB
None
Optimization terminated successfully.
         Current function value: 0.426514
         Iterations 6

=== 로지스틱 회귀 요약 ===
                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                  600
Model:                        